In [ ]:
from __future__ import annotations

from typing import Any

from tklearn.agents.core import Tool, ToolCallingAgent
from tklearn.agents.models import model_factory
from tklearn.embeddings import AutoEmbedding
from tklearn.kb import KnowledgeBase
from tklearn.nn.utils.devices import get_device

In [ ]:
kb = KnowledgeBase("wiktionary")

In [ ]:
embedder = AutoEmbedding({
    "loader": "transformers",
    "name": "sentence-transformers/all-mpnet-base-v2",
})

In [ ]:
device = get_device()
max_steps = 20

# model = model_factory(
#     client="openai",
#     model_id="openai/gpt-4o-mini",
#     api_base="https://openrouter.ai/api/v1",
#     api_key=os.environ["OPENROUTER_API_KEY"],
#     temperature=1.0,
#     seed=42,
# )

# model = model_factory(
#     "llama_cpp",
#     filename="gemma-3-12b-it-q4_0.gguf",
#     repo_id="google/gemma-3-12b-it-qat-q4_0-gguf",
#     temperature=1.0,
#     seed=42,
#     device=device,
# )

model = model_factory(
    "llama_cpp",
    filename="gpt-oss-20b-mxfp4.gguf",
    repo_id="ggml-org/gpt-oss-20b-GGUF",
    temperature=1.0,
    seed=42,
    device=device,
)

In [ ]:
def create_prompts(text: str) -> list[str]:
    mentions = kb.extract_mentions(text)
    prompts = []
    for mention in mentions:
        prompt = f"""\
Look at the mention and candidate senses below and select the most appropriate sense \
for the mention in the given context.

Context: {text}

You may choose from the candidate senses listed below. If none of the candidate senses are appropriate, \
you may indicate that no suitable sense is available by stating "wiki:None".

Mention: {mention.form}
Candidate Senses:
"""
        candidates = {}
        for candidate in mention.candidates:
            sense_id = f"wiki:{candidate.sense_id}"
            prompt += f"- Definition: {candidate.definition}\n  Sense ID: {sense_id}\n"
            candidates[sense_id] = candidate
        prompts.append({
            "text": text,
            "mention": {
                "form": mention.form,
                "span": {
                    "start": mention.span.start,
                    "end": mention.span.end,
                },
            },
            "prompt": prompt,
            "candidates": candidates,
        })
    return prompts

In [ ]:
# read examples in the data folder
import json
from pathlib import Path

data_dir = Path("../../data/examples.jsonl")

try:
    with open(data_dir, "r") as f:
        examples = [json.loads(line) for line in f]
except FileNotFoundError:
    examples = [
        {
            "text": "Python is a programming language.",
        },
        {
            "text": "The jaguar is a big cat native to the Americas.",
        },
    ]


examples

In [ ]:
CandidateDict = dict[str, Any]


class FinalAnswerTool(Tool):
    name = "final_answer"
    description = "Provides a final answer to the given problem."
    inputs = {
        "answer": {
            "type": "any",
            "description": "The final answer to the problem",
        },
        "explanation": {
            "type": "string",
            "description": "The explanation for the final answer",
        },
    }
    output_type = "any"

    def __init__(self, candidates: dict[str, CandidateDict]) -> None:
        super().__init__()
        self.candidates = candidates

    def forward(self, answer: str, explanation: str) -> Any:
        validated_answer = None
        for sense in self.candidates.keys():
            if sense in answer:
                validated_answer = answer
                break
        candidate = None
        if validated_answer in self.candidates:
            candidate = self.candidates[validated_answer]
        elif "wiki:None" in answer:
            validated_answer = "wiki:None"
        else:
            raise ValueError(
                "The final answer must include at least one of the provided senses."
            )
        return {
            "llm_answer": answer,
            "llm_explanation": explanation,
            "validated_answer": validated_answer,
            "candidate": candidate,
        }

In [ ]:
for example in examples:
    text = example["text"]
    prompts = create_prompts(text)
    for prompt in prompts:
        final_answer = FinalAnswerTool(prompt["candidates"])
        agent = ToolCallingAgent(
            tools=[final_answer],
            model=model,
            max_steps=max_steps,
            verbosity_level=0,
        )
        final_answer = agent.run(prompt["prompt"])
        print(prompt["mention"])
        print("Final Answer:", final_answer)